In [7]:
import cv2
import mediapipe as mp
import time
import datetime
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import GestureRecognizer, GestureRecognizerOptions, RunningMode
import math

In [8]:
model_path = "gesture_recognizer.task"
options = GestureRecognizerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    running_mode=RunningMode.VIDEO,  
    num_hands=1,                     
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

In [9]:
recognizer = GestureRecognizer.create_from_options(options)
cap = cv2.VideoCapture(3)
if not cap.isOpened():
    print("无法打开摄像头")
    exit()
    
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 30
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('gesture_recording.mp4', fourcc, fps, (width, height))


In [10]:
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),      # 拇指
    (0,5),(5,6),(6,7),(7,8),      # 食指
    (0,9),(9,10),(10,11),(11,12), # 中指
    (0,13),(13,14),(14,15),(15,16), # 无名指
    (0,17),(17,18),(18,19),(19,20), # 小指
    (5,9),(9,13),(13,17)           # 手掌内部连接
]

def draw_hand_landmarks(frame, hand_landmarks, color=(0,255,0), thickness=2):
    h, w, _ = frame.shape
    points = []
    for lm in hand_landmarks:
        x, y = int(lm.x * w), int(lm.y * h)
        points.append((x, y))
        cv2.circle(frame, (x, y), 4, color, -1) 
    for connection in HAND_CONNECTIONS:
        start, end = connection
        if start < len(points) and end < len(points):
            cv2.line(frame, points[start], points[end], color, thickness)

In [11]:
def distance(p1, p2) -> float:
    """计算两个关键点之间的欧氏距离"""
    return math.hypot(p1.x - p2.x, p1.y - p2.y)


def is_finger_extended(landmarks, finger: str) -> bool:
    """
    判断一根手指是否伸直。
    使用 MCP → TIP 距离与各节长度之和的比值。
    对拇指使用 IP 关节角度辅助判断。
    """
    # 手指关键点索引 (MediaPipe 标准)
    finger_indices = {
        "thumb":  [1, 2, 3, 4],   # CMC, MCP, IP, TIP
        "index":  [5, 6, 7, 8],   # MCP, PIP, DIP, TIP
        "middle": [9, 10, 11, 12],
        "ring":   [13, 14, 15, 16],
        "pinky":  [17, 18, 19, 20],
    }

    pts = [landmarks[i] for i in finger_indices[finger]]
    
    if finger == "thumb":
        # 拇指使用 MCP(2) → IP(3) → TIP(4) 的距离比值
        d23 = distance(pts[1], pts[2])  # MCP → IP
        d34 = distance(pts[2], pts[3])  # IP → TIP
        d24 = distance(pts[1], pts[3])  # MCP → TIP
        total = d23 + d34
        if total == 0:
            return False
        ratio = d24 / total
           # 拇指伸直时 ratio 通常在 0.7 以上，完全弯曲时远小于 0.7
        # 也可以结合 IP 关节角度：向量 2->3 与 3->4 的夹角
        v23 = (pts[2].x - pts[1].x, pts[2].y - pts[1].y)
        v34 = (pts[3].x - pts[2].x, pts[3].y - pts[2].y)
        dot = v23[0]*v34[0] + v23[1]*v34[1]
        norm23 = math.hypot(*v23)
        norm34 = math.hypot(*v34)
        if norm23 == 0 or norm34 == 0:
            angle = 0
        else:
            cos = max(-1.0, min(1.0, dot / (norm23 * norm34)))
            angle = math.degrees(math.acos(cos))
        print(f"拇指 ratio={ratio:.2f}, angle={angle:.1f}")
        return ratio > 0.65 and angle < 40
    else:
        # 其他四指：使用 MCP → PIP → DIP → TIP 总长度
        d01 = distance(pts[0], pts[1])  # MCP → PIP
        d12 = distance(pts[1], pts[2])  # PIP → DIP
        d23 = distance(pts[2], pts[3])  # DIP → TIP
        d03 = distance(pts[0], pts[3])  # MCP → TIP
        total = d01 + d12 + d23
        if total == 0:
            return False
        ratio = d03 / total
        # 手指伸直时 MCP 到 TIP 距离接近各节总长，阈值设为 0.75
        
        return ratio > 0.75


def are_fingers_touching(landmarks, idx1: int, idx2: int, threshold: float = 0.05) -> bool:
    """判断两个指尖是否接触"""
    return distance(landmarks[idx1], landmarks[idx2]) < threshold


def recognize_gesture(hand_landmarks):
    """
    识别数字手势 0-9，返回对应的整数；若无法识别则返回 None。
    landmarks: 包含 21 个 NormalizedLandmark 的列表，索引遵循 MediaPipe 规范。
    """
    if len(hand_landmarks) < 21:
        return None

    lm = hand_landmarks

    # 1. 获取每根手指的伸直状态
    thumb_ext = is_finger_extended(lm, "thumb")
    index_ext = is_finger_extended(lm, "index")
    middle_ext = is_finger_extended(lm, "middle")
    ring_ext = is_finger_extended(lm, "ring")
    pinky_ext = is_finger_extended(lm, "pinky")

    # 2. 特殊组合判断（优先级从上到下）
    # 7: 拇指、食指、中指指尖捏在一起
    t_i_touch = are_fingers_touching(lm, 4, 8)   # thumb - index
    t_m_touch = are_fingers_touching(lm, 4, 12)  # thumb - middle
    i_m_touch = are_fingers_touching(lm, 8, 12)  # index - middle

    if t_i_touch and t_m_touch and i_m_touch:
        # 确认无名指和小指弯曲（避免与 0 混淆）
        if not ring_ext and not pinky_ext:
            return 7

    # 0: 拇指与食指接触成圈，其余三指伸直（OK 手势）
    if t_i_touch and not thumb_ext and not index_ext:
        # 至少中指和无名指伸直即可视为 0
        if middle_ext and ring_ext:
            return 0

    # 8: 拇指与食指伸直，其他弯曲（“八”字）
    if thumb_ext and index_ext and not middle_ext and not ring_ext and not pinky_ext:
        return 8

    # 6: 拇指与小指伸直，其他弯曲（“六”）
    if thumb_ext and pinky_ext and not index_ext and not middle_ext and not ring_ext:
        return 6

    # 5: 五指全部伸直
    if all([thumb_ext, index_ext, middle_ext, ring_ext, pinky_ext]):
        return 5

    # 4: 四指伸直，拇指弯曲
    if index_ext and middle_ext and ring_ext and pinky_ext and not thumb_ext:
        return 4

    # 3: 食、中、无名指伸直，小指和拇指弯曲
    if index_ext and middle_ext and ring_ext and not pinky_ext and not thumb_ext:
        return 3

    # 2: 食、中指伸直，其他弯曲
    if index_ext and middle_ext and not ring_ext and not pinky_ext and not thumb_ext:
        return 2

    # 1: 只伸直食指
    if index_ext and not middle_ext and not ring_ext and not pinky_ext and not thumb_ext:
        return 1

    # 9 与 0（握拳）的区分：所有手指都弯曲的情况
    if not any([thumb_ext, index_ext, middle_ext, ring_ext, pinky_ext]):
        # 通过食指弯曲程度区分钩状 9 和完全握拳 0
        p5, p6, p7, p8 = lm[5], lm[6], lm[7], lm[8]
        total_len = distance(p5, p6) + distance(p6, p7) + distance(p7, p8)
        tip_to_mcp = distance(p5, p8)
        ratio = tip_to_mcp / total_len if total_len > 0 else 0

        if ratio < 0.45:
            return 0   # 握拳，也表示 0
        elif 0.45 <= ratio < 0.75:
            return 9   # 食指弯曲呈钩状
        else:
            return None

    # 无匹配结果
    return None

In [12]:

# 初始化计时器（用于传递每帧的时间戳）
frame_timestamp_ms = 0
print("开始手势识别，按 'q' 键退出")

while True:
    ret, frame = cap.read()
    if not ret:
        print("无法获取视频帧")
        break
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    # 识别当前帧中的手势
    # timestamp_ms 是自视频开始以来的毫秒数，用于 VIDEO 模式
    recognition_result = recognizer.recognize_for_video(mp_image, frame_timestamp_ms)
    frame_timestamp_ms += int(1000 / fps)  # 更新下一帧的时间戳
    # 如果检测到手部
    if recognition_result.hand_landmarks:
        for hand_landmarks in recognition_result.hand_landmarks:
            # hand_landmarks 是 21 个 NormalizedLandmark 的列表，可直接传入
            number = recognize_gesture(hand_landmarks)

            if number is not None:
                h, w, _ = frame.shape
            # 固定显示在右上角：水平距右边 60 像素，垂直距顶部 80 像素
                pos_x = w - 100        # 可根据数字宽度微调
                pos_y = 80
            # 获取该手部关键点信息（用于在手上方画文字）
                cv2.putText(frame, str(number), (pos_x, pos_y),
                            cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 0, 0), 5, cv2.LINE_AA)
            draw_hand_landmarks(frame, hand_landmarks, color=(0,255,0))

    out.write(frame)
    cv2.imshow("Gesture", frame)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break
cap.release()
out.release()
cv2.destroyAllWindows()
recognizer.close() 
print("录制完成，文件保存为 gesture_recording.mp4")


开始手势识别，按 'q' 键退出
拇指 ratio=1.00, angle=6.8
拇指 ratio=0.92, angle=48.4
拇指 ratio=0.95, angle=36.3
拇指 ratio=0.94, angle=40.5
拇指 ratio=0.96, angle=33.9
拇指 ratio=0.96, angle=32.5
拇指 ratio=0.96, angle=33.5
拇指 ratio=0.96, angle=32.3
拇指 ratio=0.96, angle=33.9
拇指 ratio=0.96, angle=31.7
拇指 ratio=0.97, angle=30.2
拇指 ratio=0.96, angle=31.6
拇指 ratio=0.97, angle=30.5
拇指 ratio=0.96, angle=31.3
拇指 ratio=0.96, angle=32.6
拇指 ratio=0.97, angle=30.4
拇指 ratio=0.97, angle=30.7
拇指 ratio=0.96, angle=31.2
拇指 ratio=0.96, angle=33.2
拇指 ratio=0.96, angle=31.6
拇指 ratio=0.96, angle=33.1
拇指 ratio=0.96, angle=32.4
拇指 ratio=0.97, angle=30.4
拇指 ratio=0.97, angle=28.1
拇指 ratio=0.97, angle=29.3
拇指 ratio=0.97, angle=28.9
拇指 ratio=0.97, angle=26.7
拇指 ratio=0.97, angle=26.1
拇指 ratio=0.98, angle=24.4
拇指 ratio=0.97, angle=26.6
拇指 ratio=0.98, angle=25.2
拇指 ratio=0.98, angle=21.7
拇指 ratio=0.98, angle=20.3
拇指 ratio=0.97, angle=26.8
拇指 ratio=0.97, angle=27.5
拇指 ratio=0.98, angle=25.2
拇指 ratio=0.97, angle=26.3
拇指 ratio=0.97, angle=2